# DPN benchmark — configuration-driven execution

Open from the repository root in a kernel with the dependencies in `requirements.txt`. By default, **Run All** previews the experiment and displays saved results without installing packages, downloading data, training, or evaluating test data.

When training is enabled, the runner automatically downloads the official prepared splits if needed, verifies the pinned SHA-256 fingerprints and applies train-only preprocessing. Accepted pruning tickets are required if structural discovery is disabled. The five-model portable selection includes LTH-MLP's own mask/rewind training.

In [ ]:
# Optional dependency installation; use the active kernel environment by default.
import sys, subprocess
from pathlib import Path
REPO = Path.cwd().resolve()
if not (REPO / 'run.py').is_file() or not (REPO / 'configs').is_dir():
    raise RuntimeError('Open this notebook from the DPN-v2 repository root.')
INSTALL_REQUIREMENTS = False
if INSTALL_REQUIREMENTS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements.txt')])
    print('Restart the kernel if packages were updated.')
else:
    print('Package installation skipped.')
print('Python:', sys.executable)

## 1. Select datasets, seeds and models

Add a dataset by adding only `configs/<new_name>/dataset.json`, with the exact prepared split source, preprocessing, MLP shape, starting singleton DPN, pruning protocol and training policies. The pipeline creates the seed and model configs later. No dataset size or model architecture is guessed.

Set `RUN_TRAINING=True` deliberately to train or resume. `ALLOW_DISCOVERY` permits expensive structural DPN pruning when tickets are missing; it does not disable LTH-MLP internal pruning. Retraining can replace imported historical CSV rows, so use a separate experiment copy if those rows must be preserved.

In [ ]:
from pathlib import Path
from src.model_factory import MODEL_NAMES
AVAILABLE_DATASETS = sorted(p.parent.name for p in Path('configs').glob('*/dataset.json'))
DATASETS = AVAILABLE_DATASETS
SEEDS = [42, 77, 53, 31, 18]
# Portable review subset: no structural pruning or maximal/pruned final training.
# Accepted pruned tickets remain required for dependency resolution.
MODELS = [
    'matched_mlp', 'matched_resnet', 'lth_mlp',
    'block_matched_dpn', 'staged_dpn',
]
# To train all seven models later: MODELS = list(MODEL_NAMES)
print('Datasets:', DATASETS)
print('Seeds:', SEEDS)
print('Models:', MODELS)
RUN_TRAINING = False
ALLOW_DISCOVERY = False
REFRESH_REPORTS = False
RUN_FROZEN_TEST = False
if RUN_TRAINING and RUN_FROZEN_TEST:
    raise ValueError('Run training and final evaluation in separate notebook executions.')
print('Model conditions:', len(DATASETS)*len(SEEDS)*len(MODELS))
print('Mode:', 'TRAIN / RESUME' if RUN_TRAINING else 'DRY RUN')

## 2. Check resources and concurrency

The scheduler uses the GPUs visible to this process. It does not use GPUs that Slurm did not allocate. Change the worker limit only if the node has enough free GPU memory.

Actual training on Roihu requires a compute allocation. Eight workers per GPU achieved high utilization in the four-GH200 Covertype pilot; other workloads may need a different limit.

In [ ]:
import os, torch, shutil
MAX_WORKERS_PER_GPU = 8  # admission is bound by measured utilization/memory
GPU_UTIL_TARGET = 0.92
GPU_MEMORY_LIMIT = 0.90
MAX_SECONDS = None  # Set this below the Slurm wall limit, for example 42000.
from src.runner import detected_accelerator
DEVICE = 'auto'
backend, visible_devices = detected_accelerator(DEVICE)
print('Detected backend:', backend)
print('CUDA available:', torch.cuda.is_available())
print('Apple Metal available:', bool(getattr(torch.backends,'mps',None) and torch.backends.mps.is_available()))
print('Visible CUDA devices:', torch.cuda.device_count())
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '(not set)'))
for gpu in range(torch.cuda.device_count()):
    properties=torch.cuda.get_device_properties(gpu)
    print(f'GPU {gpu}: {properties.name}, memory={properties.total_memory/1024**3:.1f} GiB')
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi','--query-gpu=index,name,utilization.gpu,memory.used,memory.total','--format=csv'])

## 3. Preview or run discovery, config resolution and training

The default dry run validates the selection and reports device assignments without accessing dataset arrays or launching workers. It does not establish that accepted tickets or checkpoints exist. With `RUN_TRAINING=True`, the shared runner verifies data, resolves dependencies, resumes matching checkpoints, trains unfinished models and measures isolated latency after training. Structural discovery remains off unless explicitly enabled.

In [ ]:
from src.runner import run
summary=run(REPO, DATASETS, [str(s) for s in SEEDS], MODELS,
            device=DEVICE, max_workers_per_gpu=MAX_WORKERS_PER_GPU,
            gpu_util_target=GPU_UTIL_TARGET,
            gpu_memory_limit=GPU_MEMORY_LIMIT,
            max_seconds=MAX_SECONDS, dry_run=not RUN_TRAINING,
            allow_discovery=ALLOW_DISCOVERY)
print(summary)

## 4. Create and display CSVs and figures

The `training.csv` and `pruning.csv` files hold raw, per-seed results. Validation figures show the available evidence; test figures are derived only from genuine per-seed held-out test values. If test evaluation is not frozen and complete, the figures clearly state that test results are pending.

The default displays saved results only. Set `REFRESH_REPORTS=True` in section 1 to regenerate figures and validation statistics from available saved rows (including models outside the current training selection). Imported historical rows are not newly trained checkpoints.

In [ ]:
import pandas as pd
from IPython.display import display, Image
from src.results import make_plots
for name in ('pruning', 'training', 'historical_validation_summary', 'historical_complexity_summary'):
    file = REPO / 'results' / 'csvs' / f'{name}.csv'
    if file.is_file():
        frame = pd.read_csv(file)
        print(f'{name}: {len(frame)} records (showing first 40)')
        display(frame.head(40))
if REFRESH_REPORTS:
    make_plots(REPO, DATASETS)
for dataset in DATASETS:
    for kind in ('validation', 'test'):
        file = REPO / 'results' / 'images' / f'{dataset}_{kind}_accuracy.png'
        if file.exists():
            print('Figure:', file.name)
            display(Image(filename=str(file)))

## 5. Compare MLPs and DPNs with Welch’s t-test

The script compares matched MLP, ResMLP and LTH-MLP with the four DPN families. It uses the observed seed values, not rounded table summaries. It reports two-sided p-values, 95% Welch confidence intervals and Holm-adjusted p-values. It also reports paired-seed diagnostics.

Refreshing computes validation accuracy only. The saved CSV contains one split/metric report at a time.

In [ ]:
from src.statistics import compute
file = REPO / 'results' / 'csvs' / 'welch_tests.csv'
if REFRESH_REPORTS:
    compute(REPO, split='val', metric='accuracy')
if file.is_file():
    report = pd.read_csv(file)
    print(f'Saved statistical report: {len(report)} comparisons')
    if not report.empty:
        print('Split:', report['split'].unique(), 'Metric:', report['metric'].unique())
    display(report)
else:
    print('No statistical report yet. Enable REFRESH_REPORTS to generate one.')

## 6. Optional: one-time held-out test evaluation

**Leave this disabled during architecture development.** Enable it only after all requested training and validation selections are frozen. The runner will reject later training in this result directory. It cannot use test scores to select a model.

Set the switch in section 1, with training disabled. The evaluator checks every selected completion record, fingerprint and best checkpoint before opening test observations. Imported historical validation rows alone do not qualify.

In [ ]:
if RUN_FROZEN_TEST:
    if RUN_TRAINING:
        raise ValueError('Disable training before final evaluation.')
    from src.test_evaluation import evaluate_frozen
    evaluate_frozen(REPO,DATASETS,SEEDS,MODELS,
                    device='cuda' if backend == 'cuda' else 'cpu',
                    confirm_frozen=True)
    display(pd.read_csv('results/csvs/training.csv'))
else:
    print('Test evaluation not requested; no test arrays opened by the notebook.')